# IndoMeme-XAI Revision V3 — Controlled Training Phase A
## Storage-Safe + Resume-Safe Engineering Patch

Notebook ini menggantikan notebook Phase A sebelumnya **hanya pada mekanisme penyimpanan checkpoint**.

Konfigurasi ilmiah tetap mengikuti frozen `protocol_v3.yaml` dan **tidak berubah**: dataset, split, preprocessing, architectures A/B/C/D, five matched seeds, focal loss, optimizer, scheduler, batch size, gradient accumulation, early stopping, threshold calibration, dan test firewall.

### Kebijakan checkpoint baru

```text
best_model.pt
    trainable model parameters + metadata
    dipertahankan setelah run selesai

last_resume.pt
    trainable model parameters + optimizer/scheduler/scaler/RNG
    hanya untuk resume
    dihapus setelah run COMPLETE
```

Frozen pretrained parameters direkonstruksi dari exact pinned model revisions, sehingga tidak perlu disimpan berulang pada setiap seed.

### Legacy recovery

Jika artifact lama `best.pt` / `last.pt` masih tersedia, notebook mencoba mengonversinya. Completed seed dipertahankan tanpa retraining; interrupted seed dilanjutkan dari checkpoint terakhir yang valid.

Patch ini **tidak mengubah metodologi eksperimen** dan official test tetap terkunci.

## 1. Execution selector dan path final

`RUN_VARIANT` adalah selector eksekusi, bukan hyperparameter.

Seluruh konfigurasi eksperimen lainnya dibaca langsung dari frozen `protocol_v3.yaml`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import shutil
import time
import unicodedata

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
)

# ============================================================
# SATU-SATUNYA NILAI YANG BOLEH DIUBAH ANTAR EKSEKUSI A/B/C/D.
# ============================================================
RUN_VARIANT = "C"
# Pilihan valid: "A", "B", "C", "D"
# ============================================================

if RUN_VARIANT not in {"A", "B", "C", "D"}:
    raise ValueError(
        "RUN_VARIANT harus salah satu dari A, B, C, atau D."
    )

ENGINEERING_PATCH = "storage_safe_checkpoint_v1"

PROTOCOL_ROOT = Path(
    "/kaggle/input/datasets/reynaldaryansyah/dataset"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/reynaldaryansyah/dataset/"
    "indomeme_xai_revision/indomeme_xai_revision"
)

PROTOCOL_YAML = PROTOCOL_ROOT / "protocol_v3.yaml"
PROTOCOL_SHA_FILE = PROTOCOL_ROOT / "protocol_v3.sha256"
FREEZE_RECORD = PROTOCOL_ROOT / "protocol_v3_freeze_record.json"

MANIFEST_DIR = DATA_ROOT / "manifests"
IMAGE_ROOT = DATA_ROOT / "source_snapshot" / "images"

TRAIN_MANIFEST = MANIFEST_DIR / "train_manifest.csv"
VALIDATION_MANIFEST = MANIFEST_DIR / "validation_manifest.csv"
TEST_MANIFEST = MANIFEST_DIR / "test_manifest.csv"

EXPECTED_PROTOCOL_SHA256 = (
    "d3ba1de81ef89a67946c43af996396d294d35f01a1640168bab18c8a3bec4f12"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/indomeme_xai_revision_v3/"
    "controlled_phase_a"
    f"/variant_{RUN_VARIANT}"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("RUN_VARIANT:", RUN_VARIANT)
print("ENGINEERING_PATCH:", ENGINEERING_PATCH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

RUN_VARIANT: C
ENGINEERING_PATCH: storage_safe_checkpoint_v1
OUTPUT_ROOT: /kaggle/working/indomeme_xai_revision_v3/controlled_phase_a/variant_C


## 2. Storage diagnostics dan emergency cleanup

Sebelum model dimuat, notebook menghapus hanya artifact yang aman:

- `*.tmp` dari atomic write yang gagal;
- legacy `last.pt` milik seed yang sudah memiliki `RUN_COMPLETE`, `run_summary.json`, dan legacy `best.pt` yang dapat diverifikasi.

Legacy `best.pt` baru dihapus setelah berhasil dikonversi menjadi `best_model.pt`.

In [2]:
def disk_usage_gb(path="/kaggle/working"):
    usage = shutil.disk_usage(path)
    return {
        "total_gb": usage.total / (1024 ** 3),
        "used_gb": usage.used / (1024 ** 3),
        "free_gb": usage.free / (1024 ** 3),
    }

def file_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    return path.stat().st_size / (1024 ** 3)

def compact_storage_report(root):
    files = []
    root = Path(root)
    if root.exists():
        for pattern in [
            "**/best.pt",
            "**/last.pt",
            "**/best_model.pt",
            "**/last_resume.pt",
            "**/*.tmp",
        ]:
            for path in root.glob(pattern):
                if path.is_file():
                    files.append({
                        "path": str(path),
                        "size_gb": round(file_size_gb(path), 3),
                    })
    files = sorted(files, key=lambda x: x["size_gb"], reverse=True)
    return {
        "disk": {k: round(v, 3) for k, v in disk_usage_gb().items()},
        "checkpoint_file_count": len(files),
        "largest_checkpoint_files": files[:12],
    }

print(json.dumps(compact_storage_report(OUTPUT_ROOT), indent=2, ensure_ascii=False))

# Partial atomic-save files are never valid checkpoints.
removed_tmp = []
for tmp_path in OUTPUT_ROOT.glob("**/*.tmp"):
    if tmp_path.is_file():
        removed_tmp.append(str(tmp_path))
        tmp_path.unlink()

# Completed legacy runs do not need last.pt if best.pt is valid.
freed_completed_last = []
for seed_dir in sorted(OUTPUT_ROOT.glob("seed_*")):
    if not seed_dir.is_dir():
        continue
    marker = seed_dir / "RUN_COMPLETE"
    summary = seed_dir / "run_summary.json"
    legacy_best = seed_dir / "checkpoints" / "best.pt"
    legacy_last = seed_dir / "checkpoints" / "last.pt"
    if marker.exists() and summary.exists() and legacy_best.exists() and legacy_last.exists():
        checkpoint = torch.load(legacy_best, map_location="cpu", weights_only=False)
        required = {"model", "config_hash", "variant", "seed"}
        missing = required - set(checkpoint.keys())
        if missing:
            raise RuntimeError(
                f"Tidak aman menghapus {legacy_last}: best.pt kehilangan {sorted(missing)}"
            )
        if checkpoint["config_hash"] != EXPECTED_PROTOCOL_SHA256:
            raise RuntimeError(f"Legacy best protocol hash mismatch: {legacy_best}")
        if checkpoint["variant"] != RUN_VARIANT:
            raise RuntimeError(f"Legacy best variant mismatch: {legacy_best}")
        size = file_size_gb(legacy_last)
        legacy_last.unlink()
        freed_completed_last.append({
            "seed": seed_dir.name,
            "freed_gb": round(size, 3),
        })
        del checkpoint
        gc.collect()

print(json.dumps({
    "removed_tmp_count": len(removed_tmp),
    "completed_legacy_last_removed": freed_completed_last,
    "disk_after_cleanup": {k: round(v, 3) for k, v in disk_usage_gb().items()},
}, indent=2, ensure_ascii=False))

{
  "disk": {
    "total_gb": 19.518,
    "used_gb": 0.0,
    "free_gb": 19.502
  },
  "checkpoint_file_count": 0,
  "largest_checkpoint_files": []
}
{
  "removed_tmp_count": 0,
  "completed_legacy_last_removed": [],
  "disk_after_cleanup": {
    "total_gb": 19.518,
    "used_gb": 0.0,
    "free_gb": 19.502
  }
}


## 2. Environment capture

Setiap variant menyimpan environment manifest sendiri.

Training final membutuhkan CUDA. Mixed precision mengikuti frozen protocol: FP16 pada Kaggle T4.

In [3]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak tersedia. Aktifkan Kaggle GPU accelerator."
    )

DEVICE = torch.device("cuda:0")

environment = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_variant": RUN_VARIANT,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_total_vram_gb": (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    ),
    "transformers": package_version("transformers"),
    "huggingface_hub": package_version("huggingface_hub"),
    "torchvision": package_version("torchvision"),
    "scikit_learn": package_version("scikit-learn"),
    "pandas": package_version("pandas"),
    "numpy": package_version("numpy"),
}

(OUTPUT_ROOT / "environment.json").write_text(
    json.dumps(
        environment,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        environment,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "captured_at_utc": "2026-08-09T06:33:56.393613+00:00",
  "run_variant": "C",
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "cuda_runtime": "12.8",
  "cudnn": 91002,
  "gpu_name": "Tesla T4",
  "gpu_total_vram_gb": 14.56219482421875,
  "transformers": "5.0.0",
  "huggingface_hub": "1.11.0",
  "torchvision": "0.25.0+cu128",
  "scikit_learn": "1.6.1",
  "pandas": "2.3.3",
  "numpy": "2.0.2"
}


## 3. Frozen protocol dan manifest integrity

Training dihentikan jika:

- SHA `protocol_v3.yaml` berubah;
- freeze record berbeda;
- train/validation/test manifest checksum berbeda.

Train dan validation dibaca.

Official test **tidak dibaca sebagai DataFrame**. Hanya byte-level SHA-256 yang diverifikasi.

In [4]:
import yaml

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()

required_paths = [
    PROTOCOL_YAML,
    PROTOCOL_SHA_FILE,
    FREEZE_RECORD,
    TRAIN_MANIFEST,
    VALIDATION_MANIFEST,
    TEST_MANIFEST,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"File wajib tidak ditemukan: {path}"
        )

actual_protocol_sha = sha256_file(
    PROTOCOL_YAML
)

if actual_protocol_sha != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError(
        "Frozen protocol SHA berubah. Training dihentikan."
    )

protocol = yaml.safe_load(
    PROTOCOL_YAML.read_text(
        encoding="utf-8"
    )
)

freeze_record = json.loads(
    FREEZE_RECORD.read_text(
        encoding="utf-8"
    )
)

assert (
    protocol["protocol"]["status"]
    == "FROZEN"
)

assert (
    freeze_record["protocol_yaml_sha256"]
    == EXPECTED_PROTOCOL_SHA256
)

expected_manifest_sha = (
    protocol["dataset"]["manifest_sha256"]
)

observed_manifest_sha = {
    "train": sha256_file(
        TRAIN_MANIFEST
    ),
    "validation": sha256_file(
        VALIDATION_MANIFEST
    ),
    "test": sha256_file(
        TEST_MANIFEST
    ),
}

for split_name, observed_sha in (
    observed_manifest_sha.items()
):
    if (
        observed_sha
        != expected_manifest_sha[
            split_name
        ]
    ):
        raise RuntimeError(
            f"Checksum manifest {split_name} berubah."
        )

train_df = pd.read_csv(
    TRAIN_MANIFEST
)

validation_df = pd.read_csv(
    VALIDATION_MANIFEST
)

expected_counts = (
    protocol["dataset"]["final_counts"]
)

assert (
    len(train_df)
    == int(
        expected_counts["train"]
    )
)

assert (
    len(validation_df)
    == int(
        expected_counts["validation"]
    )
)

assert (
    int(
        expected_counts["test"]
    )
    == 1005
)

print("Protocol SHA:", actual_protocol_sha)
print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Official test: checksum verified only")
print("Integrity: PASS")

Protocol SHA: d3ba1de81ef89a67946c43af996396d294d35f01a1640168bab18c8a3bec4f12
Train rows: 3402
Validation rows: 572
Official test: checksum verified only
Integrity: PASS


## 4. Frozen training configuration

Semua nilai diambil dari protocol.

Notebook akan melakukan assertion terhadap nilai penting agar perubahan tidak disengaja segera terdeteksi.

In [5]:
TRAIN_CFG = protocol[
    "training"
]

OPT_CFG = TRAIN_CFG[
    "optimizer"
]

SCHED_CFG = TRAIN_CFG[
    "scheduler"
]

THRESHOLD_CFG = protocol[
    "threshold"
]

SEEDS = [
    int(seed)
    for seed in TRAIN_CFG[
        "seeds"
    ]
]

BATCH_SIZE = int(
    TRAIN_CFG[
        "batch_size"
    ]
)

ACCUMULATION_STEPS = int(
    TRAIN_CFG[
        "gradient_accumulation_steps"
    ]
)

MAX_EPOCHS = int(
    TRAIN_CFG[
        "max_epochs"
    ]
)

PATIENCE = int(
    TRAIN_CFG[
        "early_stopping_patience"
    ]
)

GRADIENT_CLIP_NORM = float(
    TRAIN_CFG[
        "gradient_clip_norm"
    ]
)

assert SEEDS == [
    42,
    123,
    2024,
    7,
    999,
]

assert BATCH_SIZE == 32
assert ACCUMULATION_STEPS == 2
assert MAX_EPOCHS == 15
assert PATIENCE == 5
assert float(
    TRAIN_CFG["loss"]["gamma"]
) == 2.0

run_config = {
    "run_variant": RUN_VARIANT,
    "seeds": SEEDS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": ACCUMULATION_STEPS,
    "effective_batch_size": (
        BATCH_SIZE
        * ACCUMULATION_STEPS
    ),
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "loss": TRAIN_CFG["loss"],
    "optimizer": OPT_CFG,
    "scheduler": SCHED_CFG,
    "threshold": THRESHOLD_CFG,
    "protocol_sha256": actual_protocol_sha,
    "official_test": "LOCKED",
}

(OUTPUT_ROOT / "run_config.json").write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "run_variant": "C",
  "seeds": [
    42,
    123,
    2024,
    7,
    999
  ],
  "batch_size": 32,
  "gradient_accumulation_steps": 2,
  "effective_batch_size": 64,
  "max_epochs": 15,
  "early_stopping_patience": 5,
  "gradient_clip_norm": 1.0,
  "loss": {
    "name": "class-weighted focal loss",
    "gamma": 2.0,
    "class_weights": "balanced weights computed from frozen V3 train manifest",
    "label_smoothing": 0.0
  },
  "optimizer": {
    "name": "AdamW",
    "weight_decay": 0.01,
    "no_decay": [
      "bias",
      "LayerNorm",
      "layer_norm"
    ],
    "lr_encoder": 1e-05,
    "lr_fusion": 3e-05,
    "lr_head": 5e-05
  },
  "scheduler": {
    "name": "cosine",
    "warmup_ratio": 0.1,
    "step_unit": "optimizer_step"
  },
  "threshold": {
    "source": "validation only",
    "start": 0.4,
    "end": 0.65,
    "step": 0.01,
    "objective": "Macro-F1",
    "tie_break": [
      "closest to 0.5",
      "lower threshold"
    ],
    "boundary": "warn only; do not expand

## 5. OCR preprocessing dan topic representation

Implementation snapshot sama dengan Engineering Validation dan smoke run yang telah diterima.

Variant D tidak menerima explicit topic pada text maupun projection.

In [6]:
SLANG_DICT = {
    "gw":"saya","gue":"saya","gua":"saya","guwe":"saya","aq":"aku","akoh":"aku","akuh":"aku",
    "lu":"kamu","lo":"kamu","loe":"kamu","elo":"kamu","elu":"kamu","kmu":"kamu","kamuh":"kamu",
    "ane":"saya","ente":"kamu","agan":"kamu","sis":"kakak","bro":"kawan","bray":"kawan",
    "bre":"kawan","cuy":"kawan","klian":"kalian",
    "gak":"tidak","ga":"tidak","gk":"tidak","nggak":"tidak","ngga":"tidak","ngk":"tidak",
    "kgk":"tidak","tdk":"tidak","engga":"tidak","enggak":"tidak","kga":"tidak",
    "yg":"yang","dgn":"dengan","dg":"dengan","krn":"karena","krna":"karena","utk":"untuk",
    "tp":"tapi","tpi":"tapi","klo":"kalau","klau":"kalau","kl":"kalau","klu":"kalau",
    "spy":"supaya","ato":"atau","bgt":"banget","bgtt":"banget","bngt":"banget","bnr":"benar",
    "bener":"benar","emg":"memang","emang":"memang","mmg":"memang","skrg":"sekarang",
    "skrng":"sekarang","td":"tadi","tdi":"tadi","bsk":"besok","knp":"kenapa",
    "gmn":"bagaimana","gimana":"bagaimana","gmana":"bagaimana","kyk":"seperti",
    "kayak":"seperti","kek":"seperti","udh":"sudah","udah":"sudah","uda":"sudah",
    "dah":"sudah","blm":"belum","blom":"belum","lg":"lagi","lgi":"lagi","msh":"masih",
    "msi":"masih","jd":"jadi","jdi":"jadi","tau":"tahu","taw":"tahu","tw":"tahu",
    "liat":"lihat","ngeliat":"melihat","denger":"dengar","ngomong":"bicara",
    "blng":"bilang","bilg":"bilang","aja":"saja","doang":"saja","doank":"saja",
    "sih":"","deh":"","dong":"","kok":"","lah":"","loh":"","lho":"","yah":"",
    "iy":"iya","iyaa":"iya","iye":"iya","yup":"iya","yoi":"iya","org":"orang",
    "orng":"orang","orgnya":"orangnya","dr":"dari","dri":"dari","kpd":"kepada",
    "thd":"terhadap","ttg":"tentang","abis":"habis","trs":"terus","trus":"terus",
    "tmn":"teman","temen":"teman","tman":"teman","rmh":"rumah","kntr":"kantor",
    "krj":"kerja","ank":"anak","indo":"indonesia","cwo":"cowok","cwe":"cewek",
    "cwok":"cowok","cwek":"cewek","wkwk":"haha","wkwkwk":"haha","wkwkwkwk":"haha",
    "wkwkwkwkwk":"haha","hahaha":"haha","hahahaha":"haha","hehehe":"hehe",
    "hehehehe":"hehe","anjir":"anjing","anjay":"anjing","anjrit":"anjing",
    "anjr":"anjing","anying":"anjing","njir":"anjing","bgsd":"bangsat",
    "bgst":"bangsat","bjir":"anjing","bjg":"bajingan","kntl":"kontol",
    "mmk":"memek","tai":"tahi",
}

assert len(SLANG_DICT) == 149

URL_PATTERN = re.compile(
    r"(https?://\S+|www\.\S+|\S+\.(?:com|co\.id|net|org|id|tv|me)\b\S*)",
    re.IGNORECASE,
)

MENTION_PATTERN = re.compile(
    r"@\w+"
)

HASHTAG_PATTERN = re.compile(
    r"#(\w+)"
)

ALLOWED_PUNCT = re.compile(
    r"[^\w\s\.,!?\-'\"\u2019]+",
    re.UNICODE,
)

MULTI_SPACE = re.compile(
    r"\s+"
)

def safe_text(value):
    if value is None:
        return ""

    if (
        isinstance(
            value,
            float,
        )
        and pd.isna(
            value
        )
    ):
        return ""

    return str(
        value
    ).strip()

def normalize_slang(text):
    output = []

    for token in text.split():
        prefix = ""
        suffix = ""
        core = token

        while (
            core
            and not core[0].isalnum()
        ):
            prefix += core[0]
            core = core[1:]

        while (
            core
            and not core[-1].isalnum()
        ):
            suffix = (
                core[-1]
                + suffix
            )
            core = core[:-1]

        output.append(
            f"{prefix}"
            f"{SLANG_DICT.get(core.lower(), core)}"
            f"{suffix}"
        )

    return " ".join(
        output
    )

def clean_ocr(text):
    text = safe_text(
        text
    )

    if not text:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        text,
    ).lower()

    text = URL_PATTERN.sub(
        " ",
        text,
    )

    text = MENTION_PATTERN.sub(
        " ",
        text,
    )

    text = HASHTAG_PATTERN.sub(
        r"\1",
        text,
    )

    text = ALLOWED_PUNCT.sub(
        " ",
        text,
    )

    text = normalize_slang(
        text
    )

    return MULTI_SPACE.sub(
        " ",
        text,
    ).strip()

TOPIC_CATEGORIES = protocol[
    "dataset"
]["topic_categories"]

TOPIC_TO_INDEX = {
    topic: index
    for index, topic in enumerate(
        TOPIC_CATEGORIES
    )
}

def topic_to_multihot(
    value,
):
    vector = np.zeros(
        8,
        dtype=np.float32,
    )

    text = (
        safe_text(
            value
        ).lower()
        or "unknown"
    )

    parts = [
        part.strip()
        for part in text.split(",")
        if part.strip()
    ]

    if not parts:
        parts = [
            "unknown"
        ]

    for part in parts:
        if part not in TOPIC_TO_INDEX:
            raise ValueError(
                f"Topic tidak dikenal: {part!r}"
            )

        vector[
            TOPIC_TO_INDEX[
                part
            ]
        ] = 1.0

    return vector

def format_controlled_text(
    row,
    with_topic,
):
    caption = safe_text(
        row["caption"]
    )

    ocr = clean_ocr(
        row["ocr"]
    )

    if with_topic:
        topic = (
            safe_text(
                row[
                    "topic_normalized"
                ]
            ).lower()
            or "unknown"
        )

        return (
            f"[{topic}] "
            f"{caption} "
            f"[SEP] "
            f"{ocr}"
        ).strip()

    return (
        f"{caption} "
        f"[SEP] "
        f"{ocr}"
    ).strip()

assert clean_ocr(
    "GW gak liat #Kamu @abc https://x.com"
) == "saya tidak lihat kamu"

print(
    "Preprocessing implementation: PASS"
)

Preprocessing implementation: PASS


## 6. Exact model revisions

Tidak ada fallback ke `main`.

IndoBERTweet dan ViT revision harus cocok dengan frozen protocol.

In [7]:
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer,
    AutoImageProcessor,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

api = HfApi()

TEXT_SPEC = protocol[
    "models"
]["indobertweet"]

VISION_SPEC = protocol[
    "models"
]["vit"]

text_info = api.model_info(
    repo_id=TEXT_SPEC["id"],
    revision=TEXT_SPEC[
        "revision"
    ],
)

vision_info = api.model_info(
    repo_id=VISION_SPEC["id"],
    revision=VISION_SPEC[
        "revision"
    ],
)

TEXT_REVISION = text_info.sha
VISION_REVISION = vision_info.sha

if (
    TEXT_REVISION
    != TEXT_SPEC[
        "revision"
    ]
):
    raise RuntimeError(
        "Resolved IndoBERTweet revision berbeda."
    )

if not VISION_REVISION.startswith(
    VISION_SPEC[
        "revision"
    ]
):
    raise RuntimeError(
        "Resolved ViT revision berbeda."
    )

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_SPEC["id"],
    revision=TEXT_REVISION,
)

vit_processor = AutoImageProcessor.from_pretrained(
    VISION_SPEC["id"],
    revision=VISION_REVISION,
)

model_revision_report = {
    "indobertweet": {
        "model_id": TEXT_SPEC["id"],
        "requested_revision": TEXT_SPEC["revision"],
        "resolved_revision": TEXT_REVISION,
    },
    "vit": {
        "model_id": VISION_SPEC["id"],
        "requested_revision": VISION_SPEC["revision"],
        "resolved_revision": VISION_REVISION,
    },
}

(OUTPUT_ROOT / "model_revisions.json").write_text(
    json.dumps(
        model_revision_report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        model_revision_report,
        indent=2,
        ensure_ascii=False,
    )
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


{
  "indobertweet": {
    "model_id": "indolem/indobertweet-base-uncased",
    "requested_revision": "c77ff8c6b42c1d8e357e17c0e1f83c798531beef",
    "resolved_revision": "c77ff8c6b42c1d8e357e17c0e1f83c798531beef"
  },
  "vit": {
    "model_id": "google/vit-base-patch16-224-in21k",
    "requested_revision": "b456956",
    "resolved_revision": "b4569560a39a0f1af58e3ddaf17facf20ab919b0"
  }
}


## 7. Image preprocessing

Train augmentation dan validation preprocessing mengikuti frozen protocol.

`num_workers=0` dipakai secara sengaja agar checkpoint resume dapat memulihkan main-process RNG state secara tepat, termasuk augmentation randomness.

In [8]:
image_mean = (
    vit_processor.image_mean
)

image_std = (
    vit_processor.image_std
)

train_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=(
            transforms.InterpolationMode.BILINEAR
        ),
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_mean,
        std=image_std,
    ),
])

validation_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=(
            transforms.InterpolationMode.BILINEAR
        ),
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_mean,
        std=image_std,
    ),
])

print(
    "Image preprocessing: PASS"
)

Image preprocessing: PASS


## 8. Dataset class

Dataset class hanya menerima frozen train atau validation DataFrame.

Variant D menggunakan text tanpa topic prefix dan model tanpa topic projection.

In [9]:
LABEL_TO_ID = {
    "not hate": 0,
    "hate": 1,
}

MAX_LENGTH = int(
    TEXT_SPEC[
        "max_length"
    ]
)

def resolve_image_path(
    row,
):
    old_path = safe_text(
        row[
            "local_image_path"
        ]
    ).replace(
        "\\",
        "/",
    )

    filename = Path(
        old_path
    ).name

    source_split = safe_text(
        row[
            "source_split"
        ]
    )

    resolved = (
        IMAGE_ROOT
        / source_split
        / filename
    )

    if not resolved.exists():
        raise FileNotFoundError(
            f"Gambar tidak ditemukan: {resolved}"
        )

    return resolved

class ControlledDataset(
    Dataset
):
    def __init__(
        self,
        frame,
        image_transform,
        with_topic,
    ):
        self.frame = (
            frame.reset_index(
                drop=True
            ).copy()
        )

        self.image_transform = (
            image_transform
        )

        self.with_topic = bool(
            with_topic
        )

    def __len__(
        self,
    ):
        return len(
            self.frame
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.frame.iloc[
            index
        ]

        image_path = resolve_image_path(
            row
        )

        with Image.open(
            image_path
        ) as image:
            pixel_values = (
                self.image_transform(
                    image.convert(
                        "RGB"
                    )
                )
            )

        item = {
            "sample_uid": str(
                row[
                    "sample_uid"
                ]
            ),
            "text": format_controlled_text(
                row,
                with_topic=(
                    self.with_topic
                ),
            ),
            "pixel_values": pixel_values,
            "label": torch.tensor(
                LABEL_TO_ID[
                    str(
                        row[
                            "hate_final"
                        ]
                    ).strip().lower()
                ],
                dtype=torch.long,
            ),
        }

        if self.with_topic:
            item[
                "topic_features"
            ] = torch.tensor(
                topic_to_multihot(
                    row[
                        "topic_normalized"
                    ]
                ),
                dtype=torch.float32,
            )

        return item

def collate_controlled(
    batch,
):
    tokenized = tokenizer(
        [
            item[
                "text"
            ]
            for item in batch
        ],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    output = {
        "sample_uid": [
            item[
                "sample_uid"
            ]
            for item in batch
        ],
        "input_ids": tokenized[
            "input_ids"
        ],
        "attention_mask": tokenized[
            "attention_mask"
        ],
        "pixel_values": torch.stack(
            [
                item[
                    "pixel_values"
                ]
                for item in batch
            ]
        ),
        "labels": torch.stack(
            [
                item[
                    "label"
                ]
                for item in batch
            ]
        ),
    }

    if (
        "topic_features"
        in batch[0]
    ):
        output[
            "topic_features"
        ] = torch.stack(
            [
                item[
                    "topic_features"
                ]
                for item in batch
            ]
        )

    return output

## 9. Controlled model A/B/C/D

Semua variants menghasilkan 768-D multimodal representation sebelum classifier.

Variant C memakai prediction-connected bidirectional cross-attention, common-space incongruity, dan complementary gate.

In [10]:
HIDDEN_DIM = 768
NUM_HEADS = 8
ATTN_DROPOUT = 0.1
TOPIC_PROJ_DIM = 32

def set_module_trainable(
    module,
    trainable,
):
    for parameter in module.parameters():
        parameter.requires_grad = trainable

def freeze_controlled_backbones(
    text_encoder,
    vision_encoder,
):
    set_module_trainable(
        text_encoder.embeddings,
        False,
    )

    for layer in (
        text_encoder.encoder.layer[
            :2
        ]
    ):
        set_module_trainable(
            layer,
            False,
        )

    set_module_trainable(
        vision_encoder.embeddings,
        False,
    )

    for layer in (
        vision_encoder.encoder.layer[
            :2
        ]
    ):
        set_module_trainable(
            layer,
            False,
        )

def assert_freeze_policy(
    model,
):
    assert all(
        not parameter.requires_grad
        for parameter
        in model.text_encoder.embeddings.parameters()
    )

    for layer in (
        model.text_encoder.encoder.layer[
            :2
        ]
    ):
        assert all(
            not parameter.requires_grad
            for parameter
            in layer.parameters()
        )

    assert any(
        parameter.requires_grad
        for parameter
        in model.text_encoder.encoder.layer[
            2
        ].parameters()
    )

    assert all(
        not parameter.requires_grad
        for parameter
        in model.vision_encoder.embeddings.parameters()
    )

    for layer in (
        model.vision_encoder.encoder.layer[
            :2
        ]
    ):
        assert all(
            not parameter.requires_grad
            for parameter
            in layer.parameters()
        )

    assert any(
        parameter.requires_grad
        for parameter
        in model.vision_encoder.encoder.layer[
            2
        ].parameters()
    )

class ControlledMultimodalModel(
    nn.Module
):
    def __init__(
        self,
        variant,
    ):
        super().__init__()

        if variant not in {
            "A",
            "B",
            "C",
            "D",
        }:
            raise ValueError(
                f"Variant tidak dikenal: {variant}"
            )

        self.variant = variant
        self.uses_topic = (
            variant
            in {
                "A",
                "B",
                "C",
            }
        )

        self.text_encoder = (
            AutoModel.from_pretrained(
                TEXT_SPEC[
                    "id"
                ],
                revision=TEXT_REVISION,
            )
        )

        self.vision_encoder = (
            AutoModel.from_pretrained(
                VISION_SPEC[
                    "id"
                ],
                revision=VISION_REVISION,
            )
        )

        if int(
            self.text_encoder.config.hidden_size
        ) != HIDDEN_DIM:
            raise RuntimeError(
                "Unexpected IndoBERTweet hidden dimension."
            )

        if int(
            self.vision_encoder.config.hidden_size
        ) != HIDDEN_DIM:
            raise RuntimeError(
                "Unexpected ViT hidden dimension."
            )

        freeze_controlled_backbones(
            self.text_encoder,
            self.vision_encoder,
        )

        if variant == "A":
            self.late_fusion = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.fusion_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

        if variant in {
            "B",
            "D",
        }:
            self.cross_t2i = (
                nn.MultiheadAttention(
                    embed_dim=HIDDEN_DIM,
                    num_heads=NUM_HEADS,
                    dropout=ATTN_DROPOUT,
                    batch_first=True,
                )
            )

            self.cross_fusion = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.fusion_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

        if variant == "C":
            self.cross_t2i = (
                nn.MultiheadAttention(
                    embed_dim=HIDDEN_DIM,
                    num_heads=NUM_HEADS,
                    dropout=ATTN_DROPOUT,
                    batch_first=True,
                )
            )

            self.cross_i2t = (
                nn.MultiheadAttention(
                    embed_dim=HIDDEN_DIM,
                    num_heads=NUM_HEADS,
                    dropout=ATTN_DROPOUT,
                    batch_first=True,
                )
            )

            self.ca_projection = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.ca_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

            self.text_incongruity_projection = nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM,
            )

            self.image_incongruity_projection = nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM,
            )

            self.gate = nn.Linear(
                HIDDEN_DIM * 2,
                HIDDEN_DIM,
            )

            self.icam_norm = nn.LayerNorm(
                HIDDEN_DIM
            )

        if self.uses_topic:
            self.topic_projection = nn.Linear(
                8,
                TOPIC_PROJ_DIM,
            )

            classifier_input = (
                HIDDEN_DIM
                + TOPIC_PROJ_DIM
            )

        else:
            classifier_input = (
                HIDDEN_DIM
            )

        self.classifier = nn.Sequential(
            nn.Linear(
                classifier_input,
                256,
            ),
            nn.GELU(),
            nn.Dropout(
                0.3
            ),
            nn.Linear(
                256,
                2,
            ),
        )

    def semantic_text_mask(
        self,
        input_ids,
        attention_mask,
    ):
        mask = (
            attention_mask.bool().clone()
        )

        for token_id in [
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
            tokenizer.pad_token_id,
        ]:
            if token_id is not None:
                mask &= input_ids.ne(
                    int(
                        token_id
                    )
                )

        empty = (
            mask.sum(
                dim=1
            )
            .eq(
                0
            )
        )

        if empty.any():
            fallback = (
                attention_mask.bool().clone()
            )

            if tokenizer.cls_token_id is not None:
                fallback &= input_ids.ne(
                    tokenizer.cls_token_id
                )

            mask[
                empty
            ] = fallback[
                empty
            ]

        return mask

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        topic_features=None,
        return_diagnostics=False,
    ):
        text_outputs = (
            self.text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
            )
        )

        vision_outputs = (
            self.vision_encoder(
                pixel_values=pixel_values,
                return_dict=True,
            )
        )

        text_tokens = (
            text_outputs.last_hidden_state
        )

        vision_tokens = (
            vision_outputs.last_hidden_state
        )

        text_cls = (
            text_tokens[
                :,
                0,
                :,
            ]
        )

        image_cls = (
            vision_tokens[
                :,
                0,
                :,
            ]
        )

        image_patches = (
            vision_tokens[
                :,
                1:,
                :,
            ]
        )

        diagnostics = {}

        if self.variant == "A":
            representation = (
                self.fusion_norm(
                    self.late_fusion(
                        torch.cat(
                            [
                                text_cls,
                                image_cls,
                            ],
                            dim=-1,
                        )
                    )
                )
            )

        elif self.variant in {
            "B",
            "D",
        }:
            attended_image, attention_weights = (
                self.cross_t2i(
                    query=text_cls.unsqueeze(
                        1
                    ),
                    key=image_patches,
                    value=image_patches,
                    need_weights=(
                        return_diagnostics
                    ),
                    average_attn_weights=False,
                )
            )

            attended_image = (
                attended_image.squeeze(
                    1
                )
            )

            representation = (
                self.fusion_norm(
                    self.cross_fusion(
                        torch.cat(
                            [
                                text_cls,
                                attended_image,
                            ],
                            dim=-1,
                        )
                    )
                )
            )

            if return_diagnostics:
                diagnostics[
                    "t2i_attention"
                ] = attention_weights

        else:
            z_t2i, t2i_weights = (
                self.cross_t2i(
                    query=text_cls.unsqueeze(
                        1
                    ),
                    key=image_patches,
                    value=image_patches,
                    need_weights=(
                        return_diagnostics
                    ),
                    average_attn_weights=False,
                )
            )

            z_t2i = z_t2i.squeeze(
                1
            )

            z_i2t, i2t_weights = (
                self.cross_i2t(
                    query=image_cls.unsqueeze(
                        1
                    ),
                    key=text_tokens,
                    value=text_tokens,
                    key_padding_mask=(
                        ~attention_mask.bool()
                    ),
                    need_weights=(
                        return_diagnostics
                    ),
                    average_attn_weights=False,
                )
            )

            z_i2t = z_i2t.squeeze(
                1
            )

            z_ca = self.ca_norm(
                self.ca_projection(
                    torch.cat(
                        [
                            z_t2i,
                            z_i2t,
                        ],
                        dim=-1,
                    )
                )
            )

            semantic_mask = (
                self.semantic_text_mask(
                    input_ids,
                    attention_mask,
                )
            )

            text_projected = (
                F.normalize(
                    self.text_incongruity_projection(
                        text_tokens
                    ),
                    p=2,
                    dim=-1,
                )
            )

            image_projected = (
                F.normalize(
                    self.image_incongruity_projection(
                        image_patches
                    ),
                    p=2,
                    dim=-1,
                )
            )

            similarity = torch.einsum(
                "bth,bph->btp",
                text_projected,
                image_projected,
            )

            similarity = (
                similarity.masked_fill(
                    ~semantic_mask.unsqueeze(
                        -1
                    ),
                    -1e4,
                )
            )

            max_similarity = (
                similarity.max(
                    dim=1
                ).values
            )

            mismatch = (
                1.0
                - max_similarity
            )

            incongruity_weights = (
                torch.softmax(
                    mismatch,
                    dim=-1,
                )
            )

            z_inc = torch.einsum(
                "bp,bph->bh",
                incongruity_weights,
                image_projected,
            )

            gate_values = torch.sigmoid(
                self.gate(
                    torch.cat(
                        [
                            z_ca,
                            z_inc,
                        ],
                        dim=-1,
                    )
                )
            )

            representation = self.icam_norm(
                gate_values
                * z_ca
                + (
                    1.0
                    - gate_values
                )
                * z_inc
            )

            if return_diagnostics:
                diagnostics.update({
                    "t2i_attention": t2i_weights,
                    "i2t_attention": i2t_weights,
                    "incongruity_weights": incongruity_weights,
                    "gate_values": gate_values,
                })

        if self.uses_topic:
            if topic_features is None:
                raise ValueError(
                    f"Variant {self.variant} membutuhkan topic_features."
                )

            topic_embedding = (
                self.topic_projection(
                    topic_features
                )
            )

            classifier_input = torch.cat(
                [
                    representation,
                    topic_embedding,
                ],
                dim=-1,
            )

        else:
            if topic_features is not None:
                raise ValueError(
                    "Variant D tidak boleh menerima topic_features."
                )

            classifier_input = (
                representation
            )

        logits = self.classifier(
            classifier_input
        )

        if return_diagnostics:
            diagnostics[
                "representation"
            ] = representation

            return (
                logits,
                diagnostics,
            )

        return logits

## 10. Loss, metrics, optimizer groups, dan checkpoint helpers

Primary model-selection metric adalah validation Macro-F1 pada threshold 0.5.

Threshold calibration baru dilakukan setelah best checkpoint direload.

In [11]:
class_ids = (
    train_df[
        "hate_final"
    ]
    .map({
        "not hate": 0,
        "hate": 1,
    })
    .to_numpy()
)

if np.isnan(
    class_ids
).any():
    raise RuntimeError(
        "Label train tidak dapat dipetakan ke 0/1."
    )

class_ids = (
    class_ids.astype(
        np.int64
    )
)

class_counts = np.bincount(
    class_ids,
    minlength=2,
)

balanced_weights = (
    len(
        train_df
    )
    / (
        2.0
        * class_counts.astype(
            np.float64
        )
    )
)

def weighted_focal_loss(
    logits,
    targets,
    class_weights,
    gamma=2.0,
):
    log_prob = F.log_softmax(
        logits,
        dim=-1,
    )

    prob = log_prob.exp()

    target_log_prob = (
        log_prob.gather(
            1,
            targets.unsqueeze(
                1
            ),
        )
        .squeeze(
            1
        )
    )

    target_prob = (
        prob.gather(
            1,
            targets.unsqueeze(
                1
            ),
        )
        .squeeze(
            1
        )
    )

    alpha = class_weights[
        targets
    ]

    return (
        -alpha
        * (
            1.0
            - target_prob
        ).pow(
            gamma
        )
        * target_log_prob
    ).mean()

def binary_metrics(
    labels,
    probabilities,
    threshold=0.5,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities
        >= float(
            threshold
        )
    ).astype(
        np.int64
    )

    metrics = {
        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "hateful_precision": float(
            precision_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
        "hateful_recall": float(
            recall_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
        "hateful_f1": float(
            f1_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
    }

    if len(
        np.unique(
            labels
        )
    ) == 2:
        metrics[
            "roc_auc"
        ] = float(
            roc_auc_score(
                labels,
                probabilities,
            )
        )

        metrics[
            "pr_auc_ap"
        ] = float(
            average_precision_score(
                labels,
                probabilities,
            )
        )

    else:
        metrics[
            "roc_auc"
        ] = None

        metrics[
            "pr_auc_ap"
        ] = None

    return metrics

def parameter_role(
    name,
):
    if (
        name.startswith(
            "text_encoder."
        )
        or name.startswith(
            "vision_encoder."
        )
    ):
        return "encoder"

    if name.startswith(
        "classifier."
    ):
        return "head"

    return "fusion"

def is_no_decay(
    name,
):
    lower = name.lower()

    if lower.endswith(
        ".bias"
    ):
        return True

    return any(
        token in lower
        for token in [
            "layernorm",
            "layer_norm",
            ".norm.",
            "norm.weight",
        ]
    )

def build_optimizer(
    model,
):
    lr_map = {
        "encoder": float(
            OPT_CFG[
                "lr_encoder"
            ]
        ),
        "fusion": float(
            OPT_CFG[
                "lr_fusion"
            ]
        ),
        "head": float(
            OPT_CFG[
                "lr_head"
            ]
        ),
    }

    grouped = defaultdict(
        list
    )

    seen = set()

    expected = {
        id(
            parameter
        )
        for parameter in model.parameters()
        if parameter.requires_grad
    }

    for (
        name,
        parameter,
    ) in model.named_parameters():
        if not parameter.requires_grad:
            continue

        role = parameter_role(
            name
        )

        weight_decay = (
            0.0
            if is_no_decay(
                name
            )
            else float(
                OPT_CFG[
                    "weight_decay"
                ]
            )
        )

        key = (
            role,
            lr_map[
                role
            ],
            weight_decay,
        )

        grouped[
            key
        ].append(
            parameter
        )

        parameter_id = id(
            parameter
        )

        if parameter_id in seen:
            raise RuntimeError(
                f"Duplicate optimizer parameter: {name}"
            )

        seen.add(
            parameter_id
        )

    if seen != expected:
        raise RuntimeError(
            "Optimizer groups tidak mencakup tepat semua trainable parameters."
        )

    torch_groups = []
    report = []

    for (
        role,
        lr,
        weight_decay,
    ), parameters in grouped.items():
        torch_groups.append({
            "params": parameters,
            "lr": lr,
            "weight_decay": weight_decay,
        })

        report.append({
            "role": role,
            "lr": lr,
            "weight_decay": weight_decay,
            "parameter_count": int(
                sum(
                    parameter.numel()
                    for parameter in parameters
                )
            ),
            "tensor_count": int(
                len(
                    parameters
                )
            ),
        })

    optimizer = torch.optim.AdamW(
        torch_groups
    )

    return (
        optimizer,
        report,
    )

def capture_rng_state():
    return {
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "torch_cuda_rng": torch.cuda.get_rng_state_all(),
    }

def restore_rng_state(
    state,
):
    random.setstate(
        state[
            "python_rng"
        ]
    )

    np.random.set_state(
        state[
            "numpy_rng"
        ]
    )

    torch.set_rng_state(
        state[
            "torch_rng"
        ]
    )

    torch.cuda.set_rng_state_all(
        state[
            "torch_cuda_rng"
        ]
    )

def atomic_torch_save(
    state,
    destination,
):
    destination = Path(
        destination
    )

    temporary = (
        destination.with_suffix(
            destination.suffix
            + ".tmp"
        )
    )

    torch.save(
        state,
        temporary,
    )

    os.replace(
        temporary,
        destination,
    )

def set_seed(
    seed,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print(
    "Class counts:",
    class_counts.tolist(),
)

print(
    "Class weights:",
    balanced_weights.tolist(),
)

# ============================================================
# STORAGE-SAFE CHECKPOINT HELPERS
# ============================================================

def trainable_parameter_names(model):
    return {
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

def extract_trainable_state(model):
    names = trainable_parameter_names(model)
    state = model.state_dict()
    return {
        name: state[name].detach().cpu().clone()
        for name in names
    }

def extract_trainable_state_from_full_state(model, full_state):
    names = trainable_parameter_names(model)
    missing = [name for name in names if name not in full_state]
    if missing:
        raise RuntimeError(
            "Legacy full state kehilangan trainable parameters: "
            f"{missing[:10]}"
        )
    return {
        name: full_state[name].detach().cpu().clone()
        for name in names
    }

def load_trainable_state(model, trainable_state):
    expected = trainable_parameter_names(model)
    observed = set(trainable_state.keys())
    if expected != observed:
        raise RuntimeError(
            "Trainable checkpoint key mismatch. "
            f"Missing={sorted(expected-observed)[:10]}, "
            f"unexpected={sorted(observed-expected)[:10]}"
        )
    current = model.state_dict()
    current.update(trainable_state)
    model.load_state_dict(current, strict=True)

def validate_legacy_checkpoint(checkpoint, variant, seed):
    required = {"model", "config_hash", "variant", "seed", "epoch", "best_metric"}
    missing = required - set(checkpoint.keys())
    if missing:
        raise RuntimeError(f"Legacy checkpoint missing fields: {sorted(missing)}")
    if checkpoint["config_hash"] != actual_protocol_sha:
        raise RuntimeError("Legacy checkpoint protocol hash mismatch.")
    if checkpoint["variant"] != variant or int(checkpoint["seed"]) != int(seed):
        raise RuntimeError("Legacy checkpoint variant/seed mismatch.")

def best_payload_from_legacy(model, checkpoint, variant, seed):
    validate_legacy_checkpoint(checkpoint, variant, seed)
    return {
        "checkpoint_format": "trainable_state_dict_v1",
        "trainable_state": extract_trainable_state_from_full_state(model, checkpoint["model"]),
        "epoch": int(checkpoint["epoch"]),
        "best_metric": float(checkpoint["best_metric"]),
        "config_hash": actual_protocol_sha,
        "variant": variant,
        "seed": int(seed),
        "model_revisions": model_revision_report,
        "migrated_from": "legacy_full_checkpoint",
        "engineering_patch": ENGINEERING_PATCH,
    }

def resume_payload_from_legacy(model, checkpoint, variant, seed, optimizer_steps_per_epoch):
    validate_legacy_checkpoint(checkpoint, variant, seed)
    required = {
        "optimizer", "scheduler", "scaler", "no_improve",
        "python_rng", "numpy_rng", "torch_rng", "torch_cuda_rng",
    }
    missing = required - set(checkpoint.keys())
    if missing:
        raise RuntimeError(f"Legacy checkpoint tidak lengkap untuk resume: {sorted(missing)}")
    return {
        "checkpoint_format": "trainable_resume_v1",
        "trainable_state": extract_trainable_state_from_full_state(model, checkpoint["model"]),
        "optimizer": checkpoint["optimizer"],
        "scheduler": checkpoint["scheduler"],
        "scaler": checkpoint["scaler"],
        "epoch": int(checkpoint["epoch"]),
        "best_metric": float(checkpoint["best_metric"]),
        "best_epoch": checkpoint.get("best_epoch"),
        "no_improve": int(checkpoint["no_improve"]),
        "python_rng": checkpoint["python_rng"],
        "numpy_rng": checkpoint["numpy_rng"],
        "torch_rng": checkpoint["torch_rng"],
        "torch_cuda_rng": checkpoint["torch_cuda_rng"],
        "train_generator_state": checkpoint.get("train_generator_state"),
        "global_optimizer_step": int(checkpoint.get(
            "global_optimizer_step",
            int(checkpoint["epoch"]) * optimizer_steps_per_epoch,
        )),
        "config_hash": actual_protocol_sha,
        "variant": variant,
        "seed": int(seed),
        "engineering_patch": ENGINEERING_PATCH,
        "migrated_from": "legacy_full_checkpoint",
    }

def make_best_model_payload(model, epoch, best_metric, variant, seed):
    return {
        "checkpoint_format": "trainable_state_dict_v1",
        "trainable_state": extract_trainable_state(model),
        "epoch": int(epoch),
        "best_metric": float(best_metric),
        "config_hash": actual_protocol_sha,
        "variant": variant,
        "seed": int(seed),
        "model_revisions": model_revision_report,
        "engineering_patch": ENGINEERING_PATCH,
    }

def make_resume_payload(
    model, optimizer, scheduler, scaler, train_generator,
    epoch, best_metric, best_epoch, no_improve,
    global_optimizer_step, variant, seed,
):
    rng = capture_rng_state()
    return {
        "checkpoint_format": "trainable_resume_v1",
        "trainable_state": extract_trainable_state(model),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best_metric": float(best_metric),
        "best_epoch": None if best_epoch is None else int(best_epoch),
        "no_improve": int(no_improve),
        "python_rng": rng["python_rng"],
        "numpy_rng": rng["numpy_rng"],
        "torch_rng": rng["torch_rng"],
        "torch_cuda_rng": rng["torch_cuda_rng"],
        "train_generator_state": train_generator.get_state(),
        "global_optimizer_step": int(global_optimizer_step),
        "config_hash": actual_protocol_sha,
        "variant": variant,
        "seed": int(seed),
        "engineering_patch": ENGINEERING_PATCH,
    }


Class counts: [2221, 1181]
Class weights: [0.7658712291760468, 1.4403048264182896]


## 11. DataLoader factory

`num_workers=0` adalah engineering choice untuk exact epoch-boundary resume.

Train generator state disimpan di `last.pt`. Jika run terputus di tengah epoch, run dilanjutkan dari `last.pt` dan epoch yang belum selesai akan diulang dari awal sesuai frozen protocol.

In [12]:
def make_dataloaders(
    variant,
    seed,
    train_generator_state=None,
):
    with_topic = (
        variant
        in {
            "A",
            "B",
            "C",
        }
    )

    train_dataset = ControlledDataset(
        train_df,
        train_transform,
        with_topic=with_topic,
    )

    validation_dataset = ControlledDataset(
        validation_df,
        validation_transform,
        with_topic=with_topic,
    )

    train_generator = (
        torch.Generator()
    )

    train_generator.manual_seed(
        int(
            seed
        )
    )

    if (
        train_generator_state
        is not None
    ):
        train_generator.set_state(
            train_generator_state
        )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_controlled,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_controlled,
    )

    return (
        train_loader,
        validation_loader,
        train_generator,
    )

## 12. Forward helper dan validation

Validation menyimpan stable `sample_uid`, ground-truth label, dan hateful-class probability.

Prediksi validation ini diperlukan untuk threshold calibration dan ensemble phase berikutnya.

In [13]:
def batch_to_device(
    batch,
    variant,
):
    inputs = {
        "input_ids": batch[
            "input_ids"
        ].to(
            DEVICE,
            non_blocking=True,
        ),
        "attention_mask": batch[
            "attention_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        ),
        "pixel_values": batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        ),
    }

    if variant in {
        "A",
        "B",
        "C",
    }:
        inputs[
            "topic_features"
        ] = batch[
            "topic_features"
        ].to(
            DEVICE,
            non_blocking=True,
        )

    labels = batch[
        "labels"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    return (
        inputs,
        labels,
    )

@torch.no_grad()
def evaluate_validation(
    model,
    validation_loader,
    variant,
    class_weights_tensor,
):
    model.eval()

    sample_uids = []
    labels_all = []
    probabilities_all = []
    weighted_loss_sum = 0.0
    example_count = 0

    for batch in validation_loader:
        inputs, labels = (
            batch_to_device(
                batch,
                variant,
            )
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = model(
                **inputs
            )

            loss = weighted_focal_loss(
                logits,
                labels,
                class_weights=(
                    class_weights_tensor
                ),
                gamma=2.0,
            )

        probabilities = torch.softmax(
            logits.float(),
            dim=-1,
        )[
            :,
            1,
        ]

        actual_batch_size = int(
            labels.shape[
                0
            ]
        )

        weighted_loss_sum += (
            float(
                loss.detach().cpu()
            )
            * actual_batch_size
        )

        example_count += (
            actual_batch_size
        )

        sample_uids.extend(
            batch[
                "sample_uid"
            ]
        )

        labels_all.extend(
            labels.detach().cpu().tolist()
        )

        probabilities_all.extend(
            probabilities.detach().cpu().tolist()
        )

        del (
            inputs,
            labels,
            logits,
            loss,
            probabilities,
        )

    labels_array = np.asarray(
        labels_all,
        dtype=np.int64,
    )

    probabilities_array = np.asarray(
        probabilities_all,
        dtype=np.float64,
    )

    metrics = binary_metrics(
        labels_array,
        probabilities_array,
        threshold=0.5,
    )

    metrics[
        "loss"
    ] = float(
        weighted_loss_sum
        / example_count
    )

    prediction_frame = pd.DataFrame({
        "sample_uid": sample_uids,
        "label": labels_array,
        "prob_hateful": probabilities_array,
    })

    return (
        metrics,
        prediction_frame,
    )

## 13. Gradient accumulation helper

Final incomplete accumulation group menggunakan denominator berdasarkan jumlah microbatch aktual.

In [14]:
def accumulation_group_denominator(
    batch_index,
    total_batches,
    accumulation_steps,
):
    group_start = (
        batch_index
        // accumulation_steps
    ) * accumulation_steps

    group_end = min(
        group_start
        + accumulation_steps,
        total_batches,
    )

    return (
        group_end
        - group_start
    )

def accumulation_should_step(
    batch_index,
    total_batches,
    accumulation_steps,
):
    return (
        (
            (
                batch_index
                + 1
            )
            % accumulation_steps
            == 0
        )
        or (
            batch_index
            == total_batches
            - 1
        )
    )

## 14. Single-run trainer

Setiap seed mempunyai direktori sendiri:

```text
variant_X/
└── seed_<seed>/
    ├── checkpoints/
    │   ├── best.pt
    │   └── last.pt
    ├── logs/
    ├── validation/
    ├── run_summary.json
    └── RUN_COMPLETE
```

Jika `RUN_COMPLETE` ada dengan protocol hash yang benar, run dilewati.

Jika hanya `last.pt` tersedia, training resume dari epoch berikutnya.

In [15]:
def train_one_run(
    variant,
    seed,
):
    run_dir = (
        OUTPUT_ROOT
        / f"seed_{seed}"
    )

    checkpoint_dir = (
        run_dir
        / "checkpoints"
    )

    log_dir = (
        run_dir
        / "logs"
    )

    validation_dir = (
        run_dir
        / "validation"
    )

    for directory in [
        run_dir,
        checkpoint_dir,
        log_dir,
        validation_dir,
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    complete_marker = (
        run_dir
        / "RUN_COMPLETE"
    )

    summary_path = (
        run_dir
        / "run_summary.json"
    )

    best_model_path = (
        checkpoint_dir
        / "best_model.pt"
    )

    resume_path = (
        checkpoint_dir
        / "last_resume.pt"
    )

    legacy_best_path = (
        checkpoint_dir
        / "best.pt"
    )

    legacy_last_path = (
        checkpoint_dir
        / "last.pt"
    )

    epoch_log_path = (
        log_dir
        / "epoch_metrics.csv"
    )

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"START variant={variant} seed={seed}"
    )

    print(
        "=" * 72
    )

    clear_cuda()
    set_seed(
        seed
    )

    model = ControlledMultimodalModel(
        variant=variant
    ).to(
        DEVICE
    )

    assert_freeze_policy(
        model
    )

    if variant == "D":
        assert not hasattr(
            model,
            "topic_projection",
        )

        topic_parameter_names = [
            name
            for (
                name,
                _
            ) in model.named_parameters()
            if "topic" in name.lower()
        ]

        if topic_parameter_names:
            raise RuntimeError(
                "Variant D memiliki topic parameters."
            )

    # ------------------------------------------------------------
    # Legacy migration is performed after the exact architecture exists.
    # ------------------------------------------------------------
    legacy_best_checkpoint = None

    if legacy_best_path.exists() and not best_model_path.exists():
        legacy_best_checkpoint = torch.load(
            legacy_best_path,
            map_location="cpu",
            weights_only=False,
        )
        best_payload = best_payload_from_legacy(
            model,
            legacy_best_checkpoint,
            variant,
            seed,
        )
        atomic_torch_save(
            best_payload,
            best_model_path,
        )
        verified_best = torch.load(
            best_model_path,
            map_location="cpu",
            weights_only=False,
        )
        if verified_best["config_hash"] != actual_protocol_sha:
            raise RuntimeError("Migrated best_model verification failed.")
        legacy_best_path.unlink()
        del best_payload, verified_best
        gc.collect()
        print(f"[MIGRATE] {variant} seed={seed}: best.pt -> best_model.pt")

    if (
        complete_marker.exists()
        and summary_path.exists()
        and best_model_path.exists()
    ):
        completed_hash = complete_marker.read_text(encoding="utf-8").strip()
        if completed_hash == actual_protocol_sha:
            for redundant in [resume_path, legacy_last_path, legacy_best_path]:
                if redundant.exists():
                    redundant.unlink()
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            summary["engineering_patch"] = ENGINEERING_PATCH
            summary["checkpoint_format"] = "trainable_state_dict_v1"
            summary.setdefault("storage", {})["best_model_path"] = str(best_model_path)
            summary["storage"]["best_model_size_gb"] = round(file_size_gb(best_model_path), 3)
            summary["storage"]["resume_deleted_after_complete"] = True
            summary_path.write_text(
                json.dumps(summary, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
            print(
                f"[SKIP] {variant} seed={seed} sudah COMPLETE; "
                "legacy checkpoint telah dimigrasi."
            )
            if legacy_best_checkpoint is not None:
                del legacy_best_checkpoint
            del model
            clear_cuda()
            return summary

    total_parameters = int(
        sum(
            parameter.numel()
            for parameter
            in model.parameters()
        )
    )

    trainable_parameters = int(
        sum(
            parameter.numel()
            for parameter
            in model.parameters()
            if parameter.requires_grad
        )
    )

    (
        optimizer,
        optimizer_report,
    ) = build_optimizer(
        model
    )

    optimizer_report_path = (
        run_dir
        / "optimizer_groups.json"
    )

    optimizer_report_path.write_text(
        json.dumps(
            optimizer_report,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # Temporary dataloader untuk menghitung jumlah optimizer step.
    (
        train_loader,
        validation_loader,
        train_generator,
    ) = make_dataloaders(
        variant=variant,
        seed=seed,
        train_generator_state=None,
    )

    optimizer_steps_per_epoch = (
        math.ceil(
            len(
                train_loader
            )
            / ACCUMULATION_STEPS
        )
    )

    total_scheduler_steps = (
        optimizer_steps_per_epoch
        * MAX_EPOCHS
    )

    warmup_steps = int(
        math.ceil(
            total_scheduler_steps
            * float(
                SCHED_CFG[
                    "warmup_ratio"
                ]
            )
        )
    )

    scheduler = (
        get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=(
                total_scheduler_steps
            ),
        )
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )

    class_weights_tensor = torch.tensor(
        balanced_weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    start_epoch = 1
    best_metric = -float(
        "inf"
    )
    best_epoch = None
    no_improve = 0
    global_optimizer_step = 0

    epoch_records = []

    if epoch_log_path.exists():
        existing = pd.read_csv(
            epoch_log_path
        )

        epoch_records = existing.to_dict(
            orient="records"
        )

    # If an interrupted legacy run remains, convert a valid legacy last.pt.
    # The old best.pt object can be a fallback because legacy best checkpoints
    # also contained optimizer/scheduler/RNG state.
    if not resume_path.exists() and legacy_last_path.exists():
        try:
            legacy_last_checkpoint = torch.load(
                legacy_last_path,
                map_location="cpu",
                weights_only=False,
            )
            resume_payload = resume_payload_from_legacy(
                model,
                legacy_last_checkpoint,
                variant,
                seed,
                optimizer_steps_per_epoch,
            )
            atomic_torch_save(
                resume_payload,
                resume_path,
            )
            verified_resume = torch.load(
                resume_path,
                map_location="cpu",
                weights_only=False,
            )
            if verified_resume["config_hash"] != actual_protocol_sha:
                raise RuntimeError("Migrated resume verification failed.")
            legacy_last_path.unlink()
            del legacy_last_checkpoint, resume_payload, verified_resume
            gc.collect()
            print(f"[MIGRATE] {variant} seed={seed}: last.pt -> last_resume.pt")
        except Exception as exc:
            print(
                f"[WARNING] legacy last.pt tidak dapat dipakai: {exc!r}. "
                "Mencoba fallback legacy best checkpoint bila tersedia."
            )

    if (
        not resume_path.exists()
        and legacy_best_checkpoint is not None
    ):
        try:
            resume_payload = resume_payload_from_legacy(
                model,
                legacy_best_checkpoint,
                variant,
                seed,
                optimizer_steps_per_epoch,
            )
            atomic_torch_save(
                resume_payload,
                resume_path,
            )
            print(f"[MIGRATE] {variant} seed={seed}: legacy best fallback -> last_resume.pt")
            del resume_payload
            gc.collect()
        except Exception as exc:
            print(f"[WARNING] legacy best fallback tidak dapat dibuat: {exc!r}")

    if legacy_best_checkpoint is not None:
        del legacy_best_checkpoint
        gc.collect()

    if resume_path.exists():
        checkpoint = torch.load(
            resume_path,
            map_location=DEVICE,
            weights_only=False,
        )

        if checkpoint.get("checkpoint_format") != "trainable_resume_v1":
            raise RuntimeError("Unsupported resume checkpoint format.")

        if checkpoint["config_hash"] != actual_protocol_sha:
            raise RuntimeError(f"Resume checkpoint hash mismatch: {resume_path}")

        if checkpoint["variant"] != variant or int(checkpoint["seed"]) != int(seed):
            raise RuntimeError("Resume checkpoint variant/seed mismatch.")

        load_trainable_state(
            model,
            checkpoint["trainable_state"],
        )

        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        restore_rng_state(checkpoint)

        if checkpoint.get("train_generator_state") is not None:
            train_generator.set_state(checkpoint["train_generator_state"])

        start_epoch = int(checkpoint["epoch"]) + 1
        best_metric = float(checkpoint["best_metric"])
        best_epoch = checkpoint.get("best_epoch")
        if best_epoch is not None:
            best_epoch = int(best_epoch)
        no_improve = int(checkpoint["no_improve"])
        global_optimizer_step = int(checkpoint["global_optimizer_step"])

        print(
            f"Resume dari epoch {start_epoch}; "
            f"best={best_metric:.6f}; "
            f"no_improve={no_improve}; "
            f"optimizer_steps={global_optimizer_step}"
        )
        del checkpoint
        gc.collect()

    if start_epoch > MAX_EPOCHS:
        print(
            "Training epochs sudah selesai; melanjutkan finalization."
        )

    training_start = time.perf_counter()

    for epoch in range(
        start_epoch,
        MAX_EPOCHS + 1,
    ):
        if (
            no_improve
            >= PATIENCE
        ):
            print(
                f"Early stopping sebelum epoch {epoch}; "
                f"no_improve={no_improve}."
            )
            break

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        epoch_start = (
            time.perf_counter()
        )

        train_loss_sum = 0.0
        train_examples = 0

        for (
            batch_index,
            batch,
        ) in enumerate(
            train_loader
        ):
            denominator = (
                accumulation_group_denominator(
                    batch_index,
                    len(
                        train_loader
                    ),
                    ACCUMULATION_STEPS,
                )
            )

            (
                inputs,
                labels,
            ) = batch_to_device(
                batch,
                variant,
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = model(
                    **inputs
                )

                raw_loss = weighted_focal_loss(
                    logits,
                    labels,
                    class_weights=(
                        class_weights_tensor
                    ),
                    gamma=2.0,
                )

                loss_for_backward = (
                    raw_loss
                    / denominator
                )

            scaler.scale(
                loss_for_backward
            ).backward()

            actual_batch_size = int(
                labels.shape[
                    0
                ]
            )

            train_loss_sum += (
                float(
                    raw_loss.detach().cpu()
                )
                * actual_batch_size
            )

            train_examples += (
                actual_batch_size
            )

            if accumulation_should_step(
                batch_index,
                len(
                    train_loader
                ),
                ACCUMULATION_STEPS,
            ):
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=(
                        GRADIENT_CLIP_NORM
                    ),
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()

                global_optimizer_step += (
                    1
                )

            del (
                inputs,
                labels,
                logits,
                raw_loss,
                loss_for_backward,
            )

        train_loss = float(
            train_loss_sum
            / train_examples
        )

        (
            validation_metrics,
            _,
        ) = evaluate_validation(
            model=model,
            validation_loader=(
                validation_loader
            ),
            variant=variant,
            class_weights_tensor=(
                class_weights_tensor
            ),
        )

        current_metric = float(
            validation_metrics[
                "macro_f1"
            ]
        )

        improved = (
            current_metric
            > best_metric
        )

        if improved:
            best_metric = (
                current_metric
            )

            best_epoch = (
                epoch
            )

            no_improve = 0

        else:
            no_improve += 1

        if improved:
            best_payload = make_best_model_payload(
                model=model,
                epoch=epoch,
                best_metric=best_metric,
                variant=variant,
                seed=seed,
            )
            atomic_torch_save(
                best_payload,
                best_model_path,
            )
            del best_payload

        resume_payload = make_resume_payload(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            train_generator=train_generator,
            epoch=epoch,
            best_metric=best_metric,
            best_epoch=best_epoch,
            no_improve=no_improve,
            global_optimizer_step=global_optimizer_step,
            variant=variant,
            seed=seed,
        )
        atomic_torch_save(
            resume_payload,
            resume_path,
        )
        del resume_payload
        gc.collect()

        epoch_seconds = float(
            time.perf_counter()
            - epoch_start
        )

        record = {
            "variant": variant,
            "seed": int(
                seed
            ),
            "epoch": int(
                epoch
            ),
            "train_loss": float(
                train_loss
            ),
            "validation_loss": float(
                validation_metrics[
                    "loss"
                ]
            ),
            "validation_macro_f1_at_0_5": float(
                validation_metrics[
                    "macro_f1"
                ]
            ),
            "validation_hateful_precision_at_0_5": float(
                validation_metrics[
                    "hateful_precision"
                ]
            ),
            "validation_hateful_recall_at_0_5": float(
                validation_metrics[
                    "hateful_recall"
                ]
            ),
            "validation_hateful_f1_at_0_5": float(
                validation_metrics[
                    "hateful_f1"
                ]
            ),
            "validation_weighted_f1_at_0_5": float(
                validation_metrics[
                    "weighted_f1"
                ]
            ),
            "validation_accuracy_at_0_5": float(
                validation_metrics[
                    "accuracy"
                ]
            ),
            "validation_roc_auc": (
                validation_metrics[
                    "roc_auc"
                ]
            ),
            "validation_pr_auc_ap": (
                validation_metrics[
                    "pr_auc_ap"
                ]
            ),
            "best_so_far": bool(
                improved
            ),
            "best_metric_so_far": float(
                best_metric
            ),
            "no_improve": int(
                no_improve
            ),
            "optimizer_steps_completed": int(
                global_optimizer_step
            ),
            "epoch_seconds": (
                epoch_seconds
            ),
            "disk_free_gb_after_checkpoint": round(
                disk_usage_gb()["free_gb"],
                3,
            ),
        }

        epoch_records = [
            existing_record
            for existing_record
            in epoch_records
            if int(
                existing_record[
                    "epoch"
                ]
            )
            != epoch
        ]

        epoch_records.append(
            record
        )

        (
            pd.DataFrame(
                epoch_records
            )
            .sort_values(
                "epoch"
            )
            .to_csv(
                epoch_log_path,
                index=False,
            )
        )

        print(
            f"[{variant} seed={seed}] "
            f"epoch={epoch:02d} "
            f"train_loss={train_loss:.5f} "
            f"val_loss={validation_metrics['loss']:.5f} "
            f"val_macro_f1@0.5={current_metric:.5f} "
            f"best={best_metric:.5f} "
            f"no_improve={no_improve}/{PATIENCE} "
            f"free_disk={record['disk_free_gb_after_checkpoint']:.2f}GB "
            f"time={epoch_seconds:.1f}s"
        )

    training_seconds = float(
        time.perf_counter()
        - training_start
    )

    if not best_model_path.exists():
        raise RuntimeError(
            f"best_model.pt tidak ditemukan untuk {variant} seed={seed}."
        )

    best_checkpoint = torch.load(
        best_model_path,
        map_location="cpu",
        weights_only=False,
    )

    if best_checkpoint.get("checkpoint_format") != "trainable_state_dict_v1":
        raise RuntimeError("Unsupported best model checkpoint format.")

    if best_checkpoint["config_hash"] != actual_protocol_sha:
        raise RuntimeError("Best checkpoint protocol hash mismatch.")

    load_trainable_state(
        model,
        best_checkpoint["trainable_state"],
    )

    (
        best_validation_metrics_at_05,
        best_validation_predictions,
    ) = evaluate_validation(
        model=model,
        validation_loader=(
            validation_loader
        ),
        variant=variant,
        class_weights_tensor=(
            class_weights_tensor
        ),
    )

    threshold_values = np.round(
        np.arange(
            float(
                THRESHOLD_CFG[
                    "start"
                ]
            ),
            float(
                THRESHOLD_CFG[
                    "end"
                ]
            )
            + 1e-9,
            float(
                THRESHOLD_CFG[
                    "step"
                ]
            ),
        ),
        2,
    )

    threshold_rows = []

    labels_array = (
        best_validation_predictions[
            "label"
        ].to_numpy(
            dtype=np.int64
        )
    )

    probabilities_array = (
        best_validation_predictions[
            "prob_hateful"
        ].to_numpy(
            dtype=np.float64
        )
    )

    for threshold in threshold_values:
        metrics = binary_metrics(
            labels_array,
            probabilities_array,
            threshold=float(
                threshold
            ),
        )

        threshold_rows.append({
            "threshold": float(
                threshold
            ),
            "macro_f1": float(
                metrics[
                    "macro_f1"
                ]
            ),
            "hateful_f1": float(
                metrics[
                    "hateful_f1"
                ]
            ),
        })

    threshold_df = pd.DataFrame(
        threshold_rows
    )

    max_macro_f1 = float(
        threshold_df[
            "macro_f1"
        ].max()
    )

    candidate_thresholds = (
        threshold_df[
            np.isclose(
                threshold_df[
                    "macro_f1"
                ],
                max_macro_f1,
                rtol=0.0,
                atol=1e-12,
            )
        ]
        .copy()
    )

    candidate_thresholds[
        "distance_to_0_5"
    ] = (
        candidate_thresholds[
            "threshold"
        ]
        - 0.5
    ).abs()

    selected = (
        candidate_thresholds
        .sort_values(
            [
                "distance_to_0_5",
                "threshold",
            ],
            ascending=[
                True,
                True,
            ],
        )
        .iloc[
            0
        ]
    )

    selected_threshold = float(
        selected[
            "threshold"
        ]
    )

    calibrated_metrics = (
        binary_metrics(
            labels_array,
            probabilities_array,
            threshold=(
                selected_threshold
            ),
        )
    )

    boundary_warning = bool(
        selected_threshold
        in {
            float(
                THRESHOLD_CFG[
                    "start"
                ]
            ),
            float(
                THRESHOLD_CFG[
                    "end"
                ]
            ),
        }
    )

    best_validation_predictions[
        "variant"
    ] = variant

    best_validation_predictions[
        "seed"
    ] = int(
        seed
    )

    best_validation_predictions[
        "best_epoch"
    ] = int(
        best_checkpoint[
            "epoch"
        ]
    )

    best_validation_predictions[
        "selected_threshold"
    ] = (
        selected_threshold
    )

    best_validation_predictions.to_csv(
        validation_dir
        / "best_validation_predictions.csv",
        index=False,
    )

    threshold_df.to_csv(
        validation_dir
        / "threshold_calibration.csv",
        index=False,
    )

    epoch_frame = (
        pd.DataFrame(
            epoch_records
        )
        .sort_values(
            "epoch"
        )
    )

    stopped_epoch = int(
        epoch_frame[
            "epoch"
        ].max()
    )

    summary = {
        "status": "COMPLETE",
        "phase": "CONTROLLED_PHASE_A_TRAIN_VALIDATION_ONLY",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "variant": variant,
        "seed": int(
            seed
        ),
        "protocol_sha256": actual_protocol_sha,
        "engineering_patch": ENGINEERING_PATCH,
        "checkpoint_format": "trainable_state_dict_v1",
        "model_revisions": (
            model_revision_report
        ),
        "dataset": {
            "train_rows": int(
                len(
                    train_df
                )
            ),
            "validation_rows": int(
                len(
                    validation_df
                )
            ),
            "official_test": "LOCKED_NOT_LOADED",
        },
        "parameters": {
            "total": total_parameters,
            "trainable": trainable_parameters,
            "trainable_percent": float(
                100.0
                * trainable_parameters
                / total_parameters
            ),
        },
        "training": {
            "batch_size": BATCH_SIZE,
            "gradient_accumulation_steps": (
                ACCUMULATION_STEPS
            ),
            "effective_batch_size": (
                BATCH_SIZE
                * ACCUMULATION_STEPS
            ),
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "stopped_epoch": (
                stopped_epoch
            ),
            "best_epoch": int(
                best_checkpoint[
                    "epoch"
                ]
            ),
            "optimizer_steps_per_full_epoch": (
                optimizer_steps_per_epoch
            ),
            "global_optimizer_steps": int(
                global_optimizer_step
            ),
            "training_seconds_this_execution": (
                training_seconds
            ),
            "class_counts": (
                class_counts.tolist()
            ),
            "class_weights": (
                balanced_weights.tolist()
            ),
        },
        "best_validation_at_threshold_0_5": {
            key: value
            for (
                key,
                value,
            )
            in best_validation_metrics_at_05.items()
        },
        "threshold_calibration": {
            "selected_threshold": selected_threshold,
            "boundary_warning": boundary_warning,
            "calibrated_metrics": (
                calibrated_metrics
            ),
        },
        "test_firewall": {
            "test_dataframe_created": False,
            "test_dataset_created": False,
            "test_loader_created": False,
            "test_manifest_byte_hash_verified": True,
        },
        "storage": {
            "best_model_path": str(best_model_path),
            "best_model_size_gb": round(file_size_gb(best_model_path), 3),
            "resume_deleted_after_complete": True,
        },
    }

    summary_path.write_text(
        json.dumps(
            summary,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    complete_marker.write_text(
        actual_protocol_sha
        + "\n",
        encoding="utf-8",
    )

    # Completed seed retains only best_model.pt; resume state is no longer needed.
    if resume_path.exists():
        resume_path.unlink()

    for legacy_path in [legacy_best_path, legacy_last_path]:
        if legacy_path.exists():
            legacy_path.unlink()

    print(
        f"COMPLETE variant={variant} seed={seed}; "
        f"best_epoch={summary['training']['best_epoch']}; "
        f"val_macro_f1@0.5="
        f"{summary['best_validation_at_threshold_0_5']['macro_f1']:.5f}; "
        f"threshold={selected_threshold:.2f}; "
        f"best_model={file_size_gb(best_model_path):.2f}GB; "
        f"free_disk={disk_usage_gb()['free_gb']:.2f}GB"
    )

    del (
        model,
        optimizer,
        scheduler,
        scaler,
        train_loader,
        validation_loader,
        train_generator,
        class_weights_tensor,
        best_checkpoint,
        best_validation_predictions,
    )

    clear_cuda()

    return summary

## 15. Menjalankan lima seeds untuk selected variant

Semua frozen seeds dijalankan secara berurutan.

Run yang sudah memiliki `RUN_COMPLETE` dengan protocol SHA yang benar otomatis dilewati.

In [16]:
variant_run_summaries = []

variant_start = (
    time.perf_counter()
)

for seed in SEEDS:
    print(
        f"\nDisk sebelum seed {seed}: "
        f"{disk_usage_gb()['free_gb']:.2f} GB free"
    )

    summary = train_one_run(
        variant=RUN_VARIANT,
        seed=int(
            seed
        ),
    )

    variant_run_summaries.append(
        summary
    )

variant_seconds = float(
    time.perf_counter()
    - variant_start
)

print(
    f"Seluruh requested seeds untuk Variant {RUN_VARIANT} selesai."
)


Disk sebelum seed 42: 19.50 GB free

START variant=C seed=42


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

... (14 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[C seed=999] epoch=01 train_loss=0.16671 val_loss=0.17762 val_macro_f1@0.5=0.63207 best=0.63207 no_improve=0/5 free_disk=15.00GB time=119.2s
[C seed=999] epoch=02 train_loss=0.12391 val_loss=0.11948 val_macro_f1@0.5=0.76075 best=0.76075 no_improve=0/5 free_disk=15.00GB time=120.3s
[C seed=999] epoch=03 train_loss=0.10479 val_loss=0.12313 val_macro_f1@0.5=0.77692 best=0.77692 no_improve=0/5 free_disk=15.00GB time=121.2s
[C seed=999] epoch=04 train_loss=0.07924 val_loss=0.13446 val_macro_f1@0.5=0.75091 best=0.77692 no_improve=1/5 free_disk=15.00GB time=118.8s
[C seed=999] epoch=05 train_loss=0.04254 val_loss=0.20233 val_macro_f1@0.5=0.76872 best=0.77692 no_improve=2/5 free_disk=15.00GB time=118.0s
[C seed=999] epoch=06 train_loss=0.02173 val_loss=0.33742 val_macro_f1@0.5=0.73216 best=0.77692 no_improve=3/5 free_disk=15.00GB time=116.2s
[C seed=999] epoch=07 train_loss=0.00648 val_loss=0.42096 val_macro_f1@0.5=0.74395 best=0.77692 no_improve=4/5 free_disk=15.00GB time=114.8s
[C seed=999] 

## 16. Variant-level aggregation

Agregasi ini hanya menggunakan **validation**, sehingga bukan tabel hasil final paper.

Tujuannya memastikan lima seeds tersedia dan menghasilkan artifact yang lengkap sebelum Phase B official-test evaluation.

In [17]:
aggregate_rows = []

for summary in variant_run_summaries:
    val05 = (
        summary[
            "best_validation_at_threshold_0_5"
        ]
    )

    calibrated = (
        summary[
            "threshold_calibration"
        ][
            "calibrated_metrics"
        ]
    )

    aggregate_rows.append({
        "variant": summary[
            "variant"
        ],
        "seed": int(
            summary[
                "seed"
            ]
        ),
        "best_epoch": int(
            summary[
                "training"
            ][
                "best_epoch"
            ]
        ),
        "stopped_epoch": int(
            summary[
                "training"
            ][
                "stopped_epoch"
            ]
        ),
        "trainable_parameters": int(
            summary[
                "parameters"
            ][
                "trainable"
            ]
        ),
        "validation_macro_f1_at_0_5": float(
            val05[
                "macro_f1"
            ]
        ),
        "validation_hateful_f1_at_0_5": float(
            val05[
                "hateful_f1"
            ]
        ),
        "validation_roc_auc": (
            val05[
                "roc_auc"
            ]
        ),
        "validation_pr_auc_ap": (
            val05[
                "pr_auc_ap"
            ]
        ),
        "selected_threshold": float(
            summary[
                "threshold_calibration"
            ][
                "selected_threshold"
            ]
        ),
        "validation_macro_f1_calibrated": float(
            calibrated[
                "macro_f1"
            ]
        ),
        "validation_hateful_f1_calibrated": float(
            calibrated[
                "hateful_f1"
            ]
        ),
        "threshold_boundary_warning": bool(
            summary[
                "threshold_calibration"
            ][
                "boundary_warning"
            ]
        ),
    })

aggregate_df = pd.DataFrame(
    aggregate_rows
).sort_values(
    "seed"
)

aggregate_df.to_csv(
    OUTPUT_ROOT
    / "variant_validation_runs.csv",
    index=False,
)

if set(
    aggregate_df[
        "seed"
    ].tolist()
) != set(
    SEEDS
):
    raise RuntimeError(
        "Lima frozen seeds belum lengkap."
    )

numeric_summary_columns = [
    "validation_macro_f1_at_0_5",
    "validation_hateful_f1_at_0_5",
    "validation_roc_auc",
    "validation_pr_auc_ap",
    "validation_macro_f1_calibrated",
    "validation_hateful_f1_calibrated",
]

variant_statistics = {
    "variant": RUN_VARIANT,
    "n_seeds": int(
        len(
            aggregate_df
        )
    ),
    "seeds": sorted(
        [
            int(
                value
            )
            for value in aggregate_df[
                "seed"
            ].tolist()
        ]
    ),
    "protocol_sha256": actual_protocol_sha,
    "official_test": "LOCKED_NOT_LOADED",
    "variant_execution_seconds": (
        variant_seconds
    ),
    "metrics": {},
}

for column in numeric_summary_columns:
    values = pd.to_numeric(
        aggregate_df[
            column
        ],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )

    variant_statistics[
        "metrics"
    ][
        column
    ] = {
        "mean": float(
            np.mean(
                values
            )
        ),
        "std_sample": float(
            np.std(
                values,
                ddof=1,
            )
        ),
        "min": float(
            np.min(
                values
            )
        ),
        "max": float(
            np.max(
                values
            )
        ),
    }

(OUTPUT_ROOT / "variant_validation_summary.json").write_text(
    json.dumps(
        variant_statistics,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    aggregate_df.to_string(
        index=False
    )
)

print(
    json.dumps(
        variant_statistics,
        indent=2,
        ensure_ascii=False,
    )
)

variant  seed  best_epoch  stopped_epoch  trainable_parameters  validation_macro_f1_at_0_5  validation_hateful_f1_at_0_5  validation_roc_auc  validation_pr_auc_ap  selected_threshold  validation_macro_f1_calibrated  validation_hateful_f1_calibrated  threshold_boundary_warning
      C     7           3              8             151415842                    0.774359                      0.702564            0.870029              0.797040                0.48                        0.779220                          0.713217                       False
      C    42           4              9             151415842                    0.778763                      0.696379            0.868625              0.790460                0.46                        0.790551                          0.722078                       False
      C   123           4              9             151415842                    0.779330                      0.723005            0.866757              0.790104       

## 17. Final Phase-A variant acceptance

Variant dinyatakan lengkap jika:

- lima frozen seeds selesai;
- setiap seed memiliki `best.pt`, `last.pt`, `run_summary.json`, dan `RUN_COMPLETE`;
- seluruh run memakai protocol SHA yang sama;
- official test tidak pernah dibaca/dibangun sebagai evaluation loader.

In [18]:
acceptance_rows = []

for seed in SEEDS:
    run_dir = (
        OUTPUT_ROOT
        / f"seed_{seed}"
    )

    summary_path = run_dir / "run_summary.json"
    best_model_path = run_dir / "checkpoints" / "best_model.pt"
    resume_path = run_dir / "checkpoints" / "last_resume.pt"
    legacy_best = run_dir / "checkpoints" / "best.pt"
    legacy_last = run_dir / "checkpoints" / "last.pt"
    marker_path = run_dir / "RUN_COMPLETE"

    summary = json.loads(
        summary_path.read_text(
            encoding="utf-8"
        )
    )

    firewall = summary["test_firewall"]

    accepted = all([
        summary["status"] == "COMPLETE",
        summary["protocol_sha256"] == actual_protocol_sha,
        best_model_path.exists(),
        marker_path.exists(),
        marker_path.read_text(encoding="utf-8").strip() == actual_protocol_sha,
        firewall["test_dataframe_created"] is False,
        firewall["test_dataset_created"] is False,
        firewall["test_loader_created"] is False,
        firewall["test_manifest_byte_hash_verified"] is True,
        not resume_path.exists(),
        not legacy_best.exists(),
        not legacy_last.exists(),
    ])

    acceptance_rows.append({
        "variant": RUN_VARIANT,
        "seed": int(seed),
        "accepted": bool(accepted),
        "best_model_size_gb": round(file_size_gb(best_model_path), 3),
        "resume_present_after_complete": resume_path.exists(),
        "legacy_checkpoint_present": legacy_best.exists() or legacy_last.exists(),
    })

acceptance_df = pd.DataFrame(acceptance_rows)

acceptance_df.to_csv(
    OUTPUT_ROOT / "variant_acceptance_storage_safe.csv",
    index=False,
)

variant_status = (
    "PASS"
    if acceptance_df["accepted"].all()
    else "FAIL"
)

phase_a_variant_status = {
    "status": variant_status,
    "phase": "CONTROLLED_PHASE_A_TRAIN_VALIDATION_ONLY",
    "variant": RUN_VARIANT,
    "seeds": SEEDS,
    "protocol_sha256": actual_protocol_sha,
    "engineering_patch": ENGINEERING_PATCH,
    "methodology_changed": False,
    "checkpoint_policy": {
        "best_model": "trainable_state_dict_v1",
        "resume_deleted_after_complete": True,
        "legacy_full_checkpoints_removed": True,
    },
    "official_test": "LOCKED_NOT_LOADED",
    "test_dataframe_created": False,
    "test_dataset_created": False,
    "test_loader_created": False,
    "five_seed_artifacts_complete": bool(
        acceptance_df["accepted"].all()
    ),
    "storage_end": {
        key: round(value, 3)
        for key, value in disk_usage_gb().items()
    },
    "next_action": (
        "Run the same notebook for the next controlled variant. "
        "After A/B/C/D all PASS, proceed to ICAM ablations and "
        "external comparator E before opening official test."
    ),
}

(OUTPUT_ROOT / "PHASE_A_VARIANT_STATUS.json").write_text(
    json.dumps(
        phase_a_variant_status,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

if variant_status == "PASS":
    (OUTPUT_ROOT / "PHASE_A_VARIANT_PASS").write_text(
        actual_protocol_sha + "\n",
        encoding="utf-8",
    )

print(
    acceptance_df.to_string(
        index=False
    )
)

print(
    json.dumps(
        phase_a_variant_status,
        indent=2,
        ensure_ascii=False,
    )
)

if variant_status != "PASS":
    raise RuntimeError(
        f"Variant {RUN_VARIANT} belum memenuhi storage-safe Phase-A acceptance."
    )

variant  seed  accepted  best_model_size_gb  resume_present_after_complete  legacy_checkpoint_present
      C    42      True               0.564                          False                      False
      C   123      True               0.564                          False                      False
      C  2024      True               0.564                          False                      False
      C     7      True               0.564                          False                      False
      C   999      True               0.564                          False                      False
{
  "status": "PASS",
  "phase": "CONTROLLED_PHASE_A_TRAIN_VALIDATION_ONLY",
  "variant": "C",
  "seeds": [
    42,
    123,
    2024,
    7,
    999
  ],
  "protocol_sha256": "d3ba1de81ef89a67946c43af996396d294d35f01a1640168bab18c8a3bec4f12",
  "engineering_patch": "storage_safe_checkpoint_v1",
  "methodology_changed": false,
  "checkpoint_policy": {
    "best_model": "trainable_state